# OCR Debug Notebook

Use this notebook to inspect the live CS capture, the extracted digit windows, and the template scores.

Workflow:
1. Keep League visible on screen.
2. Adjust the crop settings in the config cell if needed.
3. Run the capture cell to see the live sampled region and digit windows.
4. Run the score cell to compare the captured digits against the templates, especially `0` vs `9`.


In [1]:
import ctypes
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Stats import CSOCR, capture_grayscale
from digits import TARGET_DIGITS

DIGIT_WIDTH = 10
DIGIT_HEIGHT = 12
DIGIT_BACKGROUND = 22
ocr = CSOCR()


def template_label(index):
    return "blank" if index == 10 else str(index)


def mse(a, b):
    return sum((x - y) * (x - y) for x, y in zip(a, b)) / len(a)


def shift_digit(digit_data, dx=0, dy=0, fill=DIGIT_BACKGROUND):
    shifted = []
    for y in range(DIGIT_HEIGHT):
        for x in range(DIGIT_WIDTH):
            source_x = x - dx
            source_y = y - dy
            if 0 <= source_x < DIGIT_WIDTH and 0 <= source_y < DIGIT_HEIGHT:
                shifted.append(digit_data[source_y * DIGIT_WIDTH + source_x])
            else:
                shifted.append(fill)
    return shifted


def template_scores(digit_data, allow_shift=False):
    shifts = (-1, 0, 1) if allow_shift else (0,)
    scores = []
    for index, target in enumerate(TARGET_DIGITS):
        best_mse = None
        best_shift = (0, 0)
        for dx in shifts:
            for dy in shifts:
                candidate = shift_digit(digit_data, dx=dx, dy=dy)
                score = mse(candidate, target)
                if best_mse is None or score < best_mse:
                    best_mse = score
                    best_shift = (dx, dy)
        scores.append({"index": index, "label": template_label(index), "mse": best_mse, "shift": best_shift})
    return sorted(scores, key=lambda item: item["mse"])


def render_pixels(pixels, width, height, title="", scale=10):
    rects = []
    for y in range(height):
        for x in range(width):
            value = pixels[y * width + x]
            color = f"rgb({value},{value},{value})"
            rects.append(
                f'<rect x="{x * scale}" y="{y * scale}" width="{scale}" height="{scale}" fill="{color}" />'
            )
    svg = (
        f'<svg width="{width * scale}" height="{height * scale}" viewBox="0 0 {width * scale} {height * scale}" '
        'style="border:1px solid #ccc; image-rendering: pixelated;">'
        + ''.join(rects)
        + '</svg>'
    )
    return (
        '<div style="display:inline-block; margin:8px; vertical-align:top; font-family:monospace;">'
        + f'<div style="margin-bottom:6px;">{escape(title)}</div>'
        + svg
        + '</div>'
    )


def show_gallery(items, scale=10):
    html = ''.join(render_pixels(pixels, width, height, title=title, scale=scale) for title, pixels, width, height in items)
    display(HTML(html))


def capture_cs_region(left_offset=138, right_offset=108, top=5, bottom=25):
    screen_width = ctypes.windll.user32.GetSystemMetrics(0)
    left = screen_width - left_offset
    right = screen_width - right_offset
    width = right - left
    height = bottom - top
    data = capture_grayscale(left, top, width, height)
    return {
        "left": left,
        "top": top,
        "right": right,
        "bottom": bottom,
        "width": width,
        "height": height,
        "data": data,
    }


def extract_digit(data, width, x, digit_top=3, digit_width=DIGIT_WIDTH, digit_height=DIGIT_HEIGHT):
    return ocr._extract_digit(data, width, x, digit_top=digit_top, digit_width=digit_width, digit_height=digit_height)


In [5]:
# Editable settings
left_offset = 138
right_offset = 108
top = 6
bottom = 25
digit_top = 3
x_positions = [0, 9, 19]
allow_shift = False
template_compare = [0, 9, 10]


In [6]:
capture = capture_cs_region(left_offset=left_offset, right_offset=right_offset, top=top, bottom=bottom)
print({k: capture[k] for k in ('left', 'top', 'right', 'bottom', 'width', 'height')})
print({'min': min(capture['data']), 'max': max(capture['data']), 'avg': round(sum(capture['data']) / len(capture['data']), 1)})

items = [("capture region", capture['data'], capture['width'], capture['height'])]
for x in x_positions:
    digit_data = extract_digit(capture['data'], capture['width'], x, digit_top=digit_top)
    items.append((f'digit window x={x}', digit_data, DIGIT_WIDTH, DIGIT_HEIGHT))
show_gallery(items, scale=10)


{'left': 1782, 'top': 6, 'right': 1812, 'bottom': 25, 'width': 30, 'height': 19}
{'min': 0, 'max': 237, 'avg': 73.7}


In [7]:
for x in x_positions:
    print(f'===== x={x} =====')
    digit_data = extract_digit(capture['data'], capture['width'], x, digit_top=digit_top)
    scores = template_scores(digit_data, allow_shift=allow_shift)
    for row in scores[:5]:
        print(f"{row['label']:>5}  mse={row['mse']:.1f}  shift={row['shift']}")
    compare_items = [(f'captured x={x}', digit_data, DIGIT_WIDTH, DIGIT_HEIGHT)]
    for index in template_compare:
        compare_items.append((f'template {template_label(index)}', TARGET_DIGITS[index], DIGIT_WIDTH, DIGIT_HEIGHT))
    show_gallery(compare_items, scale=12)


===== x=0 =====
    1  mse=7416.6  shift=(0, 0)
blank  mse=11056.4  shift=(0, 0)
    7  mse=12179.3  shift=(0, 0)
    5  mse=13935.3  shift=(0, 0)
    2  mse=14208.4  shift=(0, 0)


===== x=9 =====
    9  mse=9.0  shift=(0, 0)
    0  mse=6649.6  shift=(0, 0)
    8  mse=9391.5  shift=(0, 0)
    2  mse=9449.6  shift=(0, 0)
    5  mse=10849.3  shift=(0, 0)


===== x=19 =====
    9  mse=7.1  shift=(0, 0)
    0  mse=6656.5  shift=(0, 0)
    8  mse=9408.1  shift=(0, 0)
    2  mse=9455.2  shift=(0, 0)
    5  mse=10860.1  shift=(0, 0)
